# Operation HealthGuard — Audit Notebook (student starter)

> **Your role.** Chief AI Risk Officer at Klariova Health.
>
> **What this notebook does.** Orchestrates the technical audit (Task 4) by calling your implementations in `lab/`, and then writes results to `governance_portfolio.xlsx` (the single Excel deliverable).
>
> **Where the work happens.**
> - `lab/fairness_metrics.py` — fairness primitives (FNR, FPR, demographic parity, equalized odds)
> - `lab/audit.py` — subgroup, intersectional, and proxy-pathway analysis
> - `lab/shap_analysis.py` — SHAP setup and proxy-feature detection
> - `lab/kri.py` — KRI implementations
>
> **Test your code.** Before running the notebook, run the unit tests:
> ```
> pytest lab/tests/ -v
> ```
> All 34 tests must pass.

## 0 · Imports and load artifacts

In [1]:
import sys, joblib, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

# Ensure starter/ is on the path so `from lab.<module> import ...` works
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "lab").exists() else HERE.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

Project root: /Users/mkml/Documents/work/Udacity/ai-sec-eng-c4-cd15150-project-starter/starter


In [2]:
# Resolve provided artifacts — they live in starter/ regardless of where the
# notebook is being run from. The starter/ directory is where students download
# A1-A6; the solution notebook reads those same artifacts.
# Locate the data/ folder — works whether the notebook is in starter/ or solution/.
def _find_data(start: Path) -> Path:
    for cand in [start / "data", start.parent / "starter" / "data",
                 start.parent.parent / "starter" / "data",
                 start.parent / "data"]:
        if (cand / "healthguard_model.joblib").exists():
            return cand.resolve()
    raise FileNotFoundError("could not locate data/ with healthguard_model.joblib")
DATA = _find_data(ROOT)
MODEL_PATH = DATA / "healthguard_model.joblib"
TRAIN_PATH = DATA / "train.csv"
TEST_PATH  = DATA / "test.csv"

# Portfolio: write to the workbook in the same folder as this notebook.
PORTFOLIO = ROOT / "governance_portfolio.xlsx"
assert PORTFOLIO.exists(), f"Could not find {PORTFOLIO}"

bundle = joblib.load(MODEL_PATH)
model, threshold, schema = bundle["model"], bundle["decision_threshold"], bundle["feature_schema"]
train = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH)
X_train, y_train = train[schema["all"]], train["diabetes_outcome"].values
X_test,  y_test  = test[schema["all"]],  test["diabetes_outcome"].values
proba_test = model.predict_proba(X_test)[:, 1]
y_pred_test = (proba_test >= threshold).astype(int)

print(f"Model loaded; threshold = {threshold}")
print(f"Test n={len(test):,} | prevalence={y_test.mean():.3f}")
print(f"Portfolio = {PORTFOLIO}")

Model loaded; threshold = 0.26
Test n=3,000 | prevalence=0.250
Portfolio = /Users/mkml/Documents/work/Udacity/ai-sec-eng-c4-cd15150-project-starter/starter/governance_portfolio.xlsx


## 1 · Sanity-check overall performance

In [3]:
acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test)
rec = recall_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)
auc = roc_auc_score(y_test, proba_test)
print(f"  ROC AUC  : {auc:.3f}")
print(f"  Recall   : {rec:.3f}")
print(f"  Precision: {prec:.3f}")
print(f"  F1       : {f1:.3f}")
print(f"  Accuracy : {acc:.3f}")

  ROC AUC  : 0.812
  Recall   : 0.744
  Precision: 0.476
  F1       : 0.581
  Accuracy : 0.731


## 2 · Verify lab/ implementations work

Run the cell below. If you see `NotImplementedError`, your TODO functions in `lab/` are not yet implemented. Switch back to the lab/ files and implement them, then re-run.

In [4]:
from lab.fairness_metrics import (
    false_negative_rate, false_positive_rate, selection_rate,
    demographic_parity_difference, equalized_odds_difference,
)
from lab.audit import (
    compute_subgroup_metrics, find_most_harmed_subgroup, assess_proxy_pathway,
)
from lab.shap_analysis import setup_explainer, detect_proxy_features, explain_false_negatives
from lab.kri import (
    kri_subgroup_fnr_drift, kri_override_health, kri_inference_anomaly,
)

# Quick validation: compute FNR overall and on Rural + Medicaid cell
overall_fnr = false_negative_rate(y_test, y_pred_test)
print(f"Overall FNR = {overall_fnr:.3f}  (should be ~0.40)")

Overall FNR = 0.256  (should be ~0.40)


## 3 · Single-axis fairness — Task 4a

Compute subgroup metrics across each candidate axis. Look at the output and answer (in your written D4 narrative — Tab 04 of the portfolio): which axes show meaningful disparity?

In [5]:
for col in ["sex", "age_group", "region", "insurance_type"]:
    df = compute_subgroup_metrics(y_test, y_pred_test, sensitive_features=test[col])
    print(f"\n--- {col} ---")
    print(df.round(3).fillna("-"))


--- sex ---
     n_total  n_positives  selection_rate    FNR    FPR  accuracy
sex                                                              
F       1622          422           0.414  0.246  0.294     0.718
M       1378          328           0.364  0.268  0.249     0.747

--- age_group ---
           n_total  n_positives  selection_rate    FNR    FPR  accuracy
age_group                                                              
21-35          652           92           0.229  0.489  0.182     0.775
36-50          949          189           0.336  0.265  0.237     0.758
51-65          915          279           0.463  0.229  0.329     0.702
66+            484          190           0.579  0.174  0.418     0.678

--- region ---
          n_total  n_positives  selection_rate    FNR    FPR  accuracy
region                                                                
Rural         633          111           0.270  0.378  0.195     0.773
Suburban     1031          273           0.

## 4 · Intersectional fairness — Task 4b

Slice on `region × insurance_type`. Pay attention to `n_positives` — cells with very small `n_positives` (e.g. <10) are statistically unstable. Use `find_most_harmed_subgroup` with `min_n_positives=20` so the cell you escalate to leadership is one you can defend.

In [6]:
intersect = compute_subgroup_metrics(
    y_test, y_pred_test,
    sensitive_features=test[["region", "insurance_type"]])
print(intersect.round(3).fillna("-"))

                         n_total  n_positives  selection_rate    FNR    FPR  \
region   insurance_type                                                       
Rural    Medicaid            284           33           0.204  0.606  0.179   
         Medicare             64           16           0.422  0.312  0.333   
         Private             195           52           0.369  0.231  0.224   
         Uninsured            90           10           0.156  0.500  0.112   
Suburban Medicaid            185           49           0.416  0.245  0.294   
         Medicare            196           50           0.464  0.220  0.356   
         Private             572          153           0.435  0.235  0.315   
         Uninsured            78           21           0.295  0.381  0.175   
Urban    Medicaid            204           52           0.397  0.192  0.257   
         Medicare            252           64           0.437  0.172  0.303   
         Private             819          233       

In [7]:
most_harmed = find_most_harmed_subgroup(intersect, metric_name="FNR", min_n_positives=20)
print("Most-harmed subgroup (n_pos>=20):", most_harmed)

Most-harmed subgroup (n_pos>=20): {'subgroup': ('Rural', 'Medicaid'), 'metric': 'FNR', 'metric_value': 0.6060606060606061, 'n_positives': 33, 'n_total': 284, 'min_n_positives': 20, 'n_eligible': 9, 'n_suppressed': 3}


## 5 · Proxy-pathway test — Task 4e

Compare the within-region gap (Rural + Medicaid vs Rural + Private) against the same gap in a reference primary value. If within > reference, it points at insurance as the operative variable.

In [8]:
verdict = assess_proxy_pathway(
    intersect, primary_axis_value="Rural",
    secondary_axis_low="Medicaid", secondary_axis_high="Private",
    metric_name="FNR")
print(verdict)

{'metric': 'FNR', 'primary_axis_value': 'Rural', 'secondary_axis_low': 'Medicaid', 'secondary_axis_high': 'Private', 'within_gap_pp': 37.52913752913753, 'reference_gap_pp': -5.23275008253549, 'reference_primary_value': 'Urban', 'verdict': 'proxy_supported', 'rationale': 'Within Rural, Medicaid FNR exceeds Private by +37.5 pp vs -5.2 pp in Urban; the secondary axis amplifies harm beyond baseline.', 'within_n_positives': (33, 52)}


## 6 · SHAP analysis — Task 4c, 4d

In [9]:
np.random.seed(20260425)
bg_idx = np.random.choice(len(X_train), size=200, replace=False)
# setup_explainer returns the BACKGROUND transformed; we transform X_test
# separately for the actual SHAP computation
explainer, X_bg_t, feat_names = setup_explainer(model, X_train.iloc[bg_idx])
X_test_t = model.named_steps["preprocessor"].transform(X_test)
shap_values = explainer.shap_values(X_test_t)
print(f"SHAP values shape: {np.asarray(shap_values).shape}")

SHAP values shape: (3000, 22)


In [10]:
# Programmatic proxy detection — your detect_proxy_features TODO
proxy_candidates = detect_proxy_features(
    shap_values, feat_names,
    protected_feature_substrings=["region", "sex", "age_group"],
    top_k=8,
)
print(proxy_candidates)

                        feature  mean_abs_shap  mean_shap  signed_strength  \
0                  num__glucose       0.806043   0.038282         0.047494   
1                      num__bmi       0.294041  -0.005666         0.019269   
2                      num__age       0.277812  -0.005954         0.021431   
3           num__family_history       0.142103  -0.004926         0.034665   
4                 num__pedigree       0.114127   0.001880         0.016476   
5           num__blood_pressure       0.109902   0.000088         0.000797   
6              num__pregnancies       0.095965  -0.001460         0.015218   
7  cat__insurance_type_Medicaid       0.076242  -0.002009         0.026348   

   rank  
0     1  
1     2  
2     3  
3     4  
4     5  
5     6  
6     7  
7     8  


In [11]:
# Local explanations on false negatives in the most-harmed group
most_harmed_mask = (
    (test["region"] == "Rural") &
    (test["insurance_type"].isin(["Medicaid", "Uninsured"]))
).values

local_examples = explain_false_negatives(
    explainer, X_test_t, feat_names,
    y_test, y_pred_test, most_harmed_mask, n_examples=3,
)
for ex in local_examples:
    print(f"\nInstance {ex['instance_index']}: top-3 negative drivers:")
    for f, v in ex["top_negative_features"][:3]:
        print(f"   {f:<35} {v:+.3f}")


Instance 155: top-3 negative drivers:
   cat__region_Rural                   -0.271
   cat__insurance_type_Medicaid        -0.234
   num__blood_pressure                 -0.162

Instance 217: top-3 negative drivers:
   num__age                            -0.448
   cat__region_Rural                   -0.268
   num__bmi                            -0.245

Instance 225: top-3 negative drivers:
   cat__insurance_type_Uninsured       -0.339
   cat__region_Rural                   -0.272
   num__family_history                 -0.108


### 6b · Conditional SHAP — the proxy fingerprint by group

Population-mean SHAP can hide a one-hot proxy: the feature pushes predictions down for holders and up for non-holders, and the two cancel in the global mean (low `signed_strength`). Here we condition on group membership and on true positives, and print the full local story (positive vs negative sums) for the false negatives above.

In [12]:
sv = np.asarray(shap_values)
fidx = {f: j for j, f in enumerate(feat_names)}
positives = y_test == 1
rows = []
for feature, column, value in [
    ("cat__insurance_type_Medicaid", "insurance_type", "Medicaid"),
    ("cat__insurance_type_Uninsured", "insurance_type", "Uninsured"),
    ("cat__insurance_type_Private", "insurance_type", "Private"),
    ("cat__region_Rural", "region", "Rural"),
]:
    holder = (test[column] == value).values
    j = fidx[feature]
    rows.append({
        "feature": feature,
        "global mean SHAP": round(sv[:, j].mean(), 3),
        "mean SHAP | holders": round(sv[holder, j].mean(), 3),
        "mean SHAP | non-holders": round(sv[~holder, j].mean(), 3),
        "mean SHAP | holders & positives": round(sv[holder & positives, j].mean(), 3),
    })
print(pd.DataFrame(rows).set_index("feature"))

print("\nLocal false negatives (Rural x Medicaid/Uninsured):")
for ex in local_examples:
    print(f"  instance {ex['instance_index']}: sum(+)={ex['sum_shap_positive']:+.3f} "
          f"sum(-)={ex['sum_shap_negative']:+.3f} | top + {ex['top_positive_features'][0][0]} "
          f"{ex['top_positive_features'][0][1]:+.3f} | proba={proba_test[ex['instance_index']]:.3f} "
          f"(threshold {threshold})")

                               global mean SHAP  mean SHAP | holders  \
feature                                                                
cat__insurance_type_Medicaid             -0.002               -0.174   
cat__insurance_type_Uninsured            -0.000               -0.222   
cat__insurance_type_Private               0.011                0.048   
cat__region_Rural                        -0.009               -0.204   

                               mean SHAP | non-holders  \
feature                                                  
cat__insurance_type_Medicaid                     0.048   
cat__insurance_type_Uninsured                    0.018   
cat__insurance_type_Private                     -0.031   
cat__region_Rural                                0.044   

                               mean SHAP | holders & positives  
feature                                                         
cat__insurance_type_Medicaid                            -0.213  
cat__insurance_type_Uni

## 7 · KRI evaluation — Task 7 driven from your kri.py

In [13]:
kri1 = kri_subgroup_fnr_drift(y_test, y_pred_test,
                              test[["region", "insurance_type"]],
                              min_n_positives=20)
print("KRI-1 Subgroup FNR Drift:", kri1)

# For KRI-2 we use synthetic baseline numbers (production would feed real telemetry)
kri2 = kri_override_health(override_rate=0.06, structured_share=0.85)
print("\nKRI-2 Override Channel Health:", kri2)

kri3 = kri_inference_anomaly(anomaly_count=0, total_inferences=2000)
print("\nKRI-3 Inference Anomaly Rate:", kri3)

KRI-1 Subgroup FNR Drift: {'kri_id': 'KRI-1', 'kri_name': 'Subgroup FNR Drift', 'status': 'Red', 'gap_pp': 43.4, 'worst_cell': ('Rural', 'Medicaid'), 'worst_fnr': 0.6060606060606061, 'best_cell': ('Urban', 'Medicare'), 'best_fnr': 0.171875, 'n_qualifying_cells': 9, 'min_n_positives': 20, 'note': "Worst ('Rural', 'Medicaid') FNR=0.606 vs best ('Urban', 'Medicare') FNR=0.172; gap 43.4 pp across 9 cells with n_pos >= 20 (Green <= 8, Amber 8-15, Red > 15)."}

KRI-2 Override Channel Health: {'kri_id': 'KRI-2', 'kri_name': 'Override Channel Health', 'status': 'Green', 'override_rate': 0.06, 'structured_share': 0.85, 'reason': 'override rate 6.0% <= 10% and structured share 85.0% >= 80%'}

KRI-3 Inference Anomaly Rate: {'kri_id': 'KRI-3', 'kri_name': 'Inference Anomaly Rate', 'status': 'Green', 'per_1000': 0.0, 'anomaly_count': 0, 'total_inferences': 2000, 'note': '0 anomalous of 2,000 inferences = 0.00 per 1,000 (Green < 1, Amber 1-5, Red > 5).'}


## 8 · Write D4 + KRI status to the Excel portfolio

This cell writes the technical-audit summary into Tab `04_D4_Technical_Audit` and the KRI status into Tab `07_D7_KRI_Dashboard` of `governance_portfolio.xlsx`.

After this, you still need to **fill in the [HAND] tabs by hand** in Excel: D1 (regulatory work), D2 (privacy work), D3 (threat model), D5 (full risk register), D6 (most model card sections), D8 (launch decision). The notebook does NOT do regulatory work for you.

In [14]:
from openpyxl import load_workbook
wb = load_workbook(PORTFOLIO)

# === Tab 04 — Technical Audit Summary ===
ws = wb["04_D4_Technical_Audit"]
ws["B6"] = round(auc, 3)
ws["B7"] = round(rec, 3)
ws["B8"] = round(prec, 3)
ws["B9"] = round(f1, 3)
ws["B10"] = round(acc, 3)
ws["B11"] = float(threshold)
ws["B12"] = int(len(y_test))
ws["B13"] = round(float(y_test.mean()), 3)

# Region-level table
region_df = compute_subgroup_metrics(y_test, y_pred_test, test["region"])
region_rows = region_df.reset_index().sort_values("subgroup" if "subgroup" in region_df.reset_index().columns else "region")
region_rows.columns = [c if c != region_df.index.name else "region" for c in region_rows.columns]
for i, (_, row) in enumerate(region_rows.iterrows()):
    r = 18 + i
    ws.cell(row=r, column=1, value=str(row.iloc[0]))
    ws.cell(row=r, column=2, value=int(row["n_total"]))
    ws.cell(row=r, column=3, value=int(row["n_positives"]))
    ws.cell(row=r, column=4, value=round(float(row["FNR"]), 3) if pd.notna(row["FNR"]) else None)
    ws.cell(row=r, column=5, value=round(float(row["FPR"]), 3) if pd.notna(row["FPR"]) else None)
    ws.cell(row=r, column=6, value=round(float(row["selection_rate"]), 3))

# Intersectional table — top cells with n_pos >= 5
inter_rows = intersect.reset_index().sort_values("n_positives", ascending=False).head(8)
for i, (_, row) in enumerate(inter_rows.iterrows()):
    r = 27 + i
    ws.cell(row=r, column=1, value=str(row["region"]))
    ws.cell(row=r, column=2, value=str(row["insurance_type"]))
    ws.cell(row=r, column=3, value=int(row["n_total"]))
    ws.cell(row=r, column=4, value=int(row["n_positives"]))
    ws.cell(row=r, column=5, value=round(float(row["FNR"]), 3) if pd.notna(row["FNR"]) else None)
    ws.cell(row=r, column=6, value=round(float(row["selection_rate"]), 3))

# Most-harmed and verdict
ws["B37"] = str(most_harmed["subgroup"]) if most_harmed else "—"
ws["B38"] = round(float(most_harmed["metric_value"]), 3) if most_harmed else None
ws["B39"] = int(most_harmed["n_positives"]) if most_harmed else None
within = verdict.get("within_gap_pp")
ws["B42"] = f"{within:+.1f} pp" if within is not None and not np.isnan(within) else "—"
ws["B43"] = verdict["verdict"]

# Top-K SHAP feature ranking — data rows start at base4+2 = 48
for i, (_, row) in enumerate(proxy_candidates.iterrows()):
    r = 48 + i
    is_proxy = "candidate" if row["mean_shap"] < -0.01 and row["signed_strength"] >= 0.5 else "no"
    ws.cell(row=r, column=1, value=int(row["rank"]))
    ws.cell(row=r, column=2, value=str(row["feature"]))
    ws.cell(row=r, column=3, value=round(float(row["mean_abs_shap"]), 3))
    ws.cell(row=r, column=4, value=round(float(row["mean_shap"]), 3))
    ws.cell(row=r, column=5, value=round(float(row["signed_strength"]), 3))
    ws.cell(row=r, column=6, value=is_proxy)

# === Tab 07 — KRI Dashboard status ===
ws_k = wb["07_D7_KRI_Dashboard"]
for i, kri in enumerate([kri1, kri2, kri3]):
    r = 11 + i
    ws_k.cell(row=r, column=1, value=i + 1)
    ws_k.cell(row=r, column=2, value=kri["kri_id"])
    ws_k.cell(row=r, column=3, value=kri["status"])
    ws_k.cell(row=r, column=4, value=next((str(kri[k]) for k in ("gap_pp", "override_rate", "per_1000")
                                                   if kri.get(k) is not None), "—"))
    ws_k.cell(row=r, column=5, value=kri.get("note") or kri.get("reason") or "")

wb.save(PORTFOLIO)
print(f"\n✓ Wrote technical-audit + KRI status to {PORTFOLIO.name}")


✓ Wrote technical-audit + KRI status to governance_portfolio.xlsx


## 8b · Extension — counterfactual proxy test

Retrain the same classifier locally with candidate proxy features removed and compare the most-harmed cell's FNR at the shipped threshold. If dropping `insurance_type` lowers the Rural × Medicaid FNR, that strengthens the proxy verdict; if the gap persists after dropping both `insurance_type` and `region`, the residual disparity sits in the training labels themselves. Training-set prevalence by cell is printed for that check. This cell does not write to the portfolio.

In [15]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

print("Training-set prevalence by region x insurance_type:")
print(train.groupby(["region", "insurance_type"])["diabetes_outcome"].mean().unstack().round(3))

def counterfactual(drop):
    cats = [c for c in schema["categorical"] if c not in drop]
    nums = schema["numeric"]
    pipe = Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", StandardScaler(), nums),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cats)])),
        ("classifier", clone(model.named_steps["classifier"])),
    ])
    pipe.fit(train[nums + cats], y_train)
    proba = pipe.predict_proba(test[nums + cats])[:, 1]
    pred = (proba >= threshold).astype(int)
    cells = compute_subgroup_metrics(y_test, pred, test[["region", "insurance_type"]])
    regions = compute_subgroup_metrics(y_test, pred, test["region"])
    k1 = kri_subgroup_fnr_drift(y_test, pred, test[["region", "insurance_type"]], min_n_positives=20)
    return {
        "dropped": ", ".join(drop) or "(none — shipped feature set)",
        "AUC": round(roc_auc_score(y_test, proba), 3),
        "Rural×Medicaid FNR": round(cells.loc[("Rural", "Medicaid"), "FNR"], 3),
        "Rural×Private FNR": round(cells.loc[("Rural", "Private"), "FNR"], 3),
        "Rural FNR": round(regions.loc["Rural", "FNR"], 3),
        "Urban FNR": round(regions.loc["Urban", "FNR"], 3),
        "KRI-1 gap (pp)": k1["gap_pp"],
        "KRI-1": k1["status"],
    }

cf = pd.DataFrame([counterfactual([]),
                   counterfactual(["insurance_type"]),
                   counterfactual(["insurance_type", "region"])]).set_index("dropped")
cf

Training-set prevalence by region x insurance_type:


insurance_type  Medicaid  Medicare  Private  Uninsured
region                                                
Rural              0.135     0.267    0.294      0.127
Suburban           0.198     0.282    0.271      0.215
Urban              0.261     0.290    0.269      0.249


,AUC,Rural×Medicaid FNR,Rural×Private FNR,Rural FNR,Urban FNR,KRI-1 gap (pp),KRI-1
dropped,,,,,,,
(none — shipped feature set),0.812,0.606,0.231,0.378,0.227,43.4,Red
insurance_type,0.807,0.485,0.423,0.441,0.243,33.1,Red
"insurance_type, region",0.806,0.364,0.231,0.270,0.265,17.1,Red


## 9 · Next steps

You have completed the orchestrated portion of the audit. Now:

1. Open `governance_portfolio.xlsx` and check that Tab `04_D4_Technical_Audit` and Tab `07_D7_KRI_Dashboard` are populated.
2. Fill in the [HAND] tabs:
   - Tab `01_D1_EU_AI_Act` — classification + gap analysis
   - Tab `02_D2_DFD_Compliance` — annotate `A6_data_flow_diagram.png` and populate the obligation table
   - Tab `03_D3_ATLAS` — three TTPs with depth
   - Tab `05_D5_Risk_Register` — synthesize from D1–D4
   - Tab `06_D6_Model_Card` — most sections
   - Tab `08_D8_Launch_Decision` — your recommendation, evidence, conditions, sign-off
3. Run the unit tests one more time to confirm your `lab/` is clean.
4. Submit: this notebook + the lab/ folder + the populated portfolio Excel.

*— end —*